# 🛡️ Tutor Guardian — Arabic Parenting LLM Fine-Tuning (Kaggle)

Fine-tunes **Qwen2.5-3B-Instruct** on Arabic parenting Q&A using Unsloth QLoRA.
Exports a GGUF Q4_K_M file ready for Ollama.

## قبل التشغيل:
1. ارفع `qa_dataset.jsonl` كـ Kaggle Dataset باسم `tg-qa-dataset`
2. أضف الـ dataset للـ notebook من قائمة **+ Add Data**
3. فعّل الـ **T4 x2 GPU** من Settings → Accelerator
4. فعّل **Internet** من Settings
5. شغّل كل الـ cells

In [ ]:
# ── 1. Install Unsloth (Kaggle-optimized) ──────────────────────────────
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "unsloth[kaggle-new]", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-300:] if result.returncode != 0 else "✅ Unsloth installed")

In [ ]:
# ── 2. Load dataset ────────────────────────────────────────────────────
import json, os, glob

# Auto-find the jsonl file in Kaggle input
candidates = glob.glob("/kaggle/input/**/qa_dataset.jsonl", recursive=True)
if not candidates:
    candidates = glob.glob("/kaggle/input/**/*.jsonl", recursive=True)

assert candidates, "❌ لم يُعثر على qa_dataset.jsonl — أضف الـ dataset من قائمة + Add Data"

dataset_path = candidates[0]
print(f"📂 Dataset: {dataset_path}")

records = []
with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            # Only include records with meaningful output (no prompt leaking)
            if (len(r.get("output", "")) > 50 and
                "تعليمات الرد" not in r.get("output", "") and
                "educators and caregivers" not in r.get("output", "")):
                records.append(r)

print(f"✅ {len(records)} Q&A pairs loaded (after quality filter)")
if records:
    sample = records[0]
    print(f"\nSample question: {sample['instruction'][:80]}")
    print(f"Sample answer:   {sample['output'][:120]}")
    print(f"Domain: {sample.get('domain')} | Age: {sample.get('age_group')}")

In [ ]:
# ── 3. Load model with Unsloth ─────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = True,
    dtype           = None,   # auto-detect
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── 4. Add LoRA adapter ────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    lora_alpha         = 32,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    lora_dropout       = 0.05,
    bias               = "none",
    use_gradient_checkpointing = "unsloth",
    random_state       = 42,
)
model.print_trainable_parameters()

In [ ]:
# ── 5. Format dataset in Qwen chat format ─────────────────────────────
from datasets import Dataset

SYSTEM_PROMPT = """أنت مساعد تربوي ذكي للأهل العرب المسلمين.
تقدم إجابات عملية وآمنة بالعربية الفصحى الميسّرة.
أجب على آخر سؤال فقط. لا تشخّص طبياً ولا تُفتِ دينياً."""

def format_record(r):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": r["instruction"]},
        {"role": "assistant", "content": r["output"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

formatted = [format_record(r) for r in records]
dataset   = Dataset.from_dict({"text": formatted})
splits    = dataset.train_test_split(test_size=0.05, seed=42)

print(f"Train: {len(splits['train'])} | Eval: {len(splits['test'])}")
print("\nSample formatted:")
print(formatted[0][:300] + "...")

In [ ]:
# ── 6. Train ───────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = splits["train"],
    eval_dataset     = splits["test"],
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,   # effective batch = 16
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.05,
        lr_scheduler_type           = "cosine",
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        eval_strategy               = "steps",
        eval_steps                  = 50,
        save_strategy               = "steps",
        save_steps                  = 100,
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        output_dir                  = "/kaggle/working/tg-checkpoints",
        report_to                   = "none",
        max_grad_norm               = 0.3,
    ),
)

print("🚀 Starting training...")
trainer_stats = trainer.train()
print(f"\n✅ Done! Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ── 7. Quick test before export ────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_q = "بنتي عمرها 5 سنين وبتغضب كثيراً وبتعيط على أي حاجة صغيرة. إيه اللي أعمله؟"

inputs = tokenizer.apply_chat_template(
    [{"role": "system",  "content": SYSTEM_PROMPT},
     {"role": "user",    "content": test_q}],
    tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids       = inputs,
    max_new_tokens  = 300,
    temperature     = 0.3,
    top_p           = 0.9,
    repetition_penalty = 1.15,
    do_sample       = True,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(f"Q: {test_q}")
print(f"\nA: {response}")

In [ ]:
# ── 8. Export GGUF Q4_K_M (Unsloth built-in, no llama.cpp needed) ──────
GGUF_PATH = "/kaggle/working/tg-tutor-v1"

print("💾 Saving GGUF Q4_K_M...")
model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer,
    quantization_method = "q4_k_m",
)

import os, glob
gguf_files = glob.glob(f"{GGUF_PATH}*.gguf")
if gguf_files:
    gguf_file = gguf_files[0]
    size_mb   = os.path.getsize(gguf_file) / 1024 / 1024
    print(f"\n✅ GGUF saved: {gguf_file} ({size_mb:.0f} MB)")
else:
    print(f"Files in {GGUF_PATH}:")
    print(os.listdir(GGUF_PATH))

In [ ]:
# ── 9. Write Ollama Modelfile ──────────────────────────────────────────
modelfile = """FROM ./tg-tutor-v1-Q4_K_M.gguf

TEMPLATE \"\"\"{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
{{ .Response }}<|im_end|>
\"\"\"

SYSTEM \"\"\"أنت مساعد تربوي ذكي للأهل العرب المسلمين.
تقدم إجابات عملية وآمنة بالعربية الفصحى الميسّرة.
أجب على آخر سؤال فقط. لا تشخّص طبياً ولا تُفتِ دينياً.\"\"\"

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.15
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
"""

with open("/kaggle/working/Modelfile.tg-tutor", "w", encoding="utf-8") as f:
    f.write(modelfile)

print("✅ Modelfile saved")
print("\n📥 Download these files from Output tab:")
print("   tg-tutor-v1-Q4_K_M.gguf  (~1.8 GB)")
print("   Modelfile.tg-tutor")
print("\n🚀 Then on your home server:")
print("   ollama create tg-tutor:v2 -f Modelfile.tg-tutor")